In [ ]:
pip install torch torchvision numpy matplotlib scikit-learn

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import time
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


In [9]:
mean, std = (0.507, 0.487, 0.441), (0.267, 0.256, 0.276)
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4),
                               transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(mean, std)])
test_tf  = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=train_tf)
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

100%|██████████| 169M/169M [22:43<00:00, 124kB/s]


Train batches: 391, Test batches: 79


In [10]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class ResNet18(nn.Module):
    def __init__(self, num_classes=100, width=64):
        super(ResNet18, self).__init__()
        self.in_channels = width

        self.conv1 = nn.Conv2d(3, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(BasicBlock, width, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, width * 2 , 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, width * 4, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, width * 8, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(8*width, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out
for w in (64, 32):
    m = ResNet18(num_classes=100, width=w)
    out = m(torch.randn(2, 3, 32, 32))
    print(w, out.shape, sum(p.numel() for p in m.parameters()))

64 torch.Size([2, 100]) 11220132
32 torch.Size([2, 100]) 2820740


In [11]:
import os, json, random
from torch.utils.flop_counter import FlopCounterMode
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/distill'
os.makedirs(SAVE_DIR, exist_ok=True)

# config (same for every run)
EPOCHS, LR, T, ALPHA = 30, 0.1, 4.0, 0.5
SEEDS = [0, 1, 2]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def count_flops(model, size=(1, 3, 32, 32)):
    model.eval()
    x = torch.randn(*size, device=next(model.parameters()).device)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    return fc.get_total_flops()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast('cuda', dtype=torch.float16):
            pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def train(model, train_loader, test_loader, epochs, loss_fn, lr=LR):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')
    history = []
    for epoch in range(epochs):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(x)
                loss = loss_fn(out, y, x)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item() * x.size(0)
            correct += out.argmax(1).eq(y).sum().item()
            total += y.size(0)
        sched.step()
        row = {'epoch': epoch + 1, 'loss': run_loss / total, 'train_acc': 100 * correct / total}
        if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
            row['test_acc'] = evaluate(model, test_loader)
        history.append(row)
        print(row)
    return {'test_acc': history[-1]['test_acc'], 'history': history}   # final epoch, not best

def ce_loss(out, y, x):
    return F.cross_entropy(out, y)

def make_kd_loss(teacher, T=T, alpha=ALPHA):
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    def kd_loss(out, y, x):
        with torch.no_grad():
            t = teacher(x)                       # same augmented batch as the student
        ce = F.cross_entropy(out, y)
        kl = F.kl_div(F.log_softmax(out.float() / T, dim=1), F.softmax(t.float() / T, dim=1), reduction='batchmean') * (T * T)
        return alpha * ce + (1 - alpha) * kl
    return kd_loss

In [12]:
RESULTS = f'{SAVE_DIR}/results.json'
results = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}

def run(name, width, loss_fn_builder, seed):
    key = f'{name}_s{seed}'
    if key in results:                                    # resume after a disconnect
        print('skip', key, results[key]['test_acc']); return
    set_seed(seed)                                        # seed BEFORE building the model
    model = ResNet18(num_classes=100, width=width).to(device)
    out = train(model, train_loader, test_loader, EPOCHS, loss_fn_builder())
    out.update(params=count_params(model), flops=count_flops(model))
    results[key] = out
    torch.save(model.state_dict(), f'{SAVE_DIR}/{key}.pth')
    json.dump(results, open(RESULTS, 'w'))
    return model

In [14]:
EPOCHS = 1
run('debug_teacher', 64, lambda: ce_loss, seed=0)
results.pop('debug_teacher_s0')
EPOCHS = 30

NameError: name 'train' is not defined

In [15]:
# Teacher: trained once
run('teacher', 64, lambda: ce_loss, seed=0)
teacher = ResNet18(num_classes=100, width=64).to(device)
teacher.load_state_dict(torch.load(f'{SAVE_DIR}/teacher_s0.pth'))
print(results['teacher_s0']['test_acc'], results['teacher_s0']['params'], results['teacher_s0']['flops'])

skip teacher_s0 74.33
74.33 11220132 1110937600


In [ ]:
for seed in SEEDS[:1]:
    run('baseline', 32, lambda: ce_loss, seed)
    run('distilled', 32, lambda: make_kd_loss(teacher), seed)

skip baseline_s0 71.35
skip distilled_s0 73.46


In [ ]:
for seed in SEEDS[1:]:
    run('baseline', 32, lambda: ce_loss, seed)
    run('distilled', 32, lambda: make_kd_loss(teacher), seed)

skip baseline_s1 70.8
skip distilled_s1 73.86
skip baseline_s2 71.84
skip distilled_s2 73.77


In [19]:
def summarize(prefix):
    accs = [results[f'{prefix}_s{s}']['test_acc'] for s in SEEDS]
    return np.mean(accs), np.std(accs, ddof=1)   # ddof=1: sample std, since n=3

for p in ('baseline', 'distilled'):
    print(p, '%.2f ± %.2f' % summarize(p))

baseline 71.33 ± 0.52
distilled 73.70 ± 0.21


In [16]:
def load(width, name):
    m = ResNet18(num_classes=100, width=width).to(device)
    m.load_state_dict(torch.load(f'{SAVE_DIR}/{name}.pth', map_location=device))
    return m.eval()

teacher = load(64, 'teacher_s0')

In [17]:
@torch.no_grad()
def benchmark(model, batch, warmup=20, iters=200):
    model.eval()
    x = torch.randn(batch, 3, 32, 32, device=device)
    for _ in range(warmup):
        model(x)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        model(x)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    return {'p50_ms': np.percentile(times, 50),
            'p95_ms': np.percentile(times, 95),
            'peak_mem_MB': torch.cuda.max_memory_allocated() / 1024**2}

@torch.no_grad()
def agreement(a, b, loader):
    a.eval(); b.eval()
    same = total = 0
    for x, _ in loader:
        x = x.to(device)
        with torch.autocast('cuda', dtype=torch.float16):
            same += (a(x).argmax(1) == b(x).argmax(1)).sum().item()
        total += x.size(0)
    return 100.0 * same / total

In [20]:
eval_out = {'gpu': torch.cuda.get_device_name(0), 'agreement': {}, 'bench': {}}

# Agreement with teacher
for s in SEEDS:
    baseline, distilled = load(32, f'baseline_s{s}'), load(32, f'distilled_s{s}')
    a_b = agreement(baseline, teacher, test_loader)
    a_d = agreement(distilled, teacher, test_loader)
    eval_out['agreement'][s] = {'baseline': a_b, 'distilled': a_d}
    print(f'seed {s} | baseline-teacher {a_b:.2f}% | distilled-teacher {a_d:.2f}%')
    del baseline, distilled

# Latency and memory
print(eval_out['gpu'])
for name, width in [('teacher_s0', 64), ('baseline_s0', 32)]:
    torch.cuda.empty_cache()
    base = torch.cuda.memory_allocated()
    m = load(width, name)
    for b in (1, 256):
        r = benchmark(m, b)
        r['peak_mem_MB'] -= base / 1024**2            # this model only
        r = {k: round(float(v), 2) for k, v in r.items()}
        eval_out['bench'][f'{name}_b{b}'] = r
        print(name, 'batch', b, r)
    del m

# FLOPs and ratios
tf, sf = results['teacher_s0']['flops'], results['baseline_s0']['flops']
eval_out['flops'] = {'teacher': tf, 'student': sf}
print(f'FLOPs teacher {tf/1e9:.3f} G | student {sf/1e9:.3f} G | ratio {tf/sf:.2f}x')
for b in (1, 256):
    t, s_ = eval_out['bench'][f'teacher_s0_b{b}'], eval_out['bench'][f'baseline_s0_b{b}']
    print(f'batch {b} speedup p50 {t["p50_ms"]/s_["p50_ms"]:.2f}x | '
          f'memory {t["peak_mem_MB"]/s_["peak_mem_MB"]:.2f}x')

# Save to Drive
for p in ('baseline', 'distilled'):
    eval_out[f'{p}_acc_mean_std'] = [float(v) for v in summarize(p)]
json.dump(eval_out, open(f'{SAVE_DIR}/eval_results.json', 'w'), indent=2)
print('saved', f'{SAVE_DIR}/eval_results.json')

seed 0 | baseline-teacher 76.33% | distilled-teacher 80.70%
seed 1 | baseline-teacher 75.92% | distilled-teacher 80.68%
seed 2 | baseline-teacher 76.96% | distilled-teacher 80.64%
Tesla T4
teacher_s0 batch 1 {'p50_ms': 3.3, 'p95_ms': 4.43, 'peak_mem_MB': 50.93}
teacher_s0 batch 256 {'p50_ms': 106.13, 'p95_ms': 109.05, 'peak_mem_MB': 593.35}
baseline_s0 batch 1 {'p50_ms': 2.2, 'p95_ms': 2.76, 'peak_mem_MB': 16.97}
baseline_s0 batch 256 {'p50_ms': 27.62, 'p95_ms': 28.08, 'peak_mem_MB': 249.3}
FLOPs teacher 1.111 G | student 0.279 G | ratio 3.99x
batch 1 speedup p50 1.50x | memory 3.00x
batch 256 speedup p50 3.84x | memory 2.38x
saved /content/drive/MyDrive/distill/eval_results.json
